# Klasifikasi DemogPairs Menggunakan ViT (Wajah) & Logistic Regression

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
features = joblib.load('features/demogpairs_vit-face.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [LogisticRegression(random_state=42)],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__max_iter': [500, 1000],
        'classifier__solver': ['lbfgs', 'saga'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

LogisticRegression: 96 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_lr_vit-face_",
    results_path="results/demogpairs_lr_vit-face_"
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: LogisticRegression


{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 500, 'classifier__solver': 'newton-cg', 'pca': None, 'scaler': 'MinMaxScaler'}


Accuracy  : 0.9060185185185186
Precision : 0.9060356756958051
Recall    : 0.9060185185185187
F1 Score  : 0.9059460424855371
               precision    recall  f1-score   support

Asian_Females     0.9014    0.8889    0.8951       360
  Asian_Males     0.8856    0.9028    0.8941       360
Black_Females     0.9086    0.8833    0.8958       360
  Black_Males     0.9162    0.9111    0.9136       360
White_Females     0.9056    0.9056    0.9056       360
  White_Males     0.9189    0.9444    0.9315       360

     accuracy                         0.9060      2160
    macro avg     0.9060    0.9060    0.9059      2160
 weighted avg     0.9060    0.9060    0.9059      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9652777777777778,0.9014084507042254,0.8888888888888888,0.8951048951048951,360
Asian_Males,0.9643518518518519,0.885558583106267,0.9027777777777778,0.8940852819807428,360
Black_Females,0.9657407407407408,0.9085714285714286,0.8833333333333333,0.895774647887324,360
Black_Males,0.9712962962962963,0.9162011173184358,0.9111111111111111,0.9136490250696379,360
White_Females,0.9685185185185186,0.9055555555555556,0.9055555555555556,0.9055555555555556,360
White_Males,0.9768518518518519,0.918918918918919,0.9444444444444444,0.9315068493150684,360


Confusion matrix saved: images\cm_lr_vit-face_LogisticRegression.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               320                13                 9                 2                14                 2
         Asian_Males                11               325                 4                 7                 0                13
       Black_Females                 7                 5               318                14                15                 1
         Black_Males                 0                12                10               328                 1                 9
       White_Females                17                 2                 9                 1               326                 5
         White_Males                 0                10                 0                 6                 4               340


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
LogisticRegression,models/clf_demogpairs_lr_vit-face_LogisticRegression.pkl,"{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 500, 'classifier__solver': 'newton-cg', 'pca': None, 'scaler': 'MinMaxScaler'}",0.9060185185185186,0.9059460424855371,0.9060356756958051,0.9060185185185187,270


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_lr_vit-face_LogisticRegression.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 3199.0,
 'days': 0,
 'hours': 0,
 'minutes': 53,
 'seconds': 19.0,
 'text': '0 hari 0 jam 53 menit 19.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 36780.0,
 'days': 0,
 'hours': 10,
 'minutes': 13,
 'seconds': 0.0,
 'text': '0 hari 10 jam 13 menit 0.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 2000, 'classifier__solver': 'newton-cg', 'pca': None, 'scaler': 'MinMaxScaler'}",0.9126,0.9068,0.8976,0.901,0.908,0.9052,0.9052,0.9055,0.9052,22.7103
2,"{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 1000, 'classifier__solver': 'newton-cg', 'pca': None, 'scaler': 'MinMaxScaler'}",0.9126,0.9068,0.8976,0.901,0.908,0.9052,0.9052,0.9055,0.9052,21.437
3,"{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 500, 'classifier__solver': 'newton-cg', 'pca': None, 'scaler': 'MinMaxScaler'}",0.9126,0.9068,0.8976,0.901,0.908,0.9052,0.9052,0.9055,0.9052,23.2741
4,"{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': 'MinMaxScaler'}",0.912,0.9057,0.897,0.9005,0.9091,0.9049,0.9048,0.9052,0.9049,41.2131
...,...,...,...,...,...,...,...,...,...,...,...
267,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 2000, 'classifier__solver': 'lbfgs', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8663,0.8443,0.842,0.8513,0.8455,0.8499,0.849,0.8503,0.8499,1.2131
268,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 500, 'classifier__solver': 'newton-cg', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8663,0.8443,0.842,0.8507,0.8455,0.8498,0.8489,0.8502,0.8498,1.0182
269,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 2000, 'classifier__solver': 'newton-cg', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8663,0.8443,0.842,0.8507,0.8455,0.8498,0.8489,0.8502,0.8498,1.2292
270,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'newton-cg', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8663,0.8443,0.842,0.8507,0.8455,0.8498,0.8489,0.8502,0.8498,1.2262
